https://zhuanlan.zhihu.com/p/595902651



你好！我很乐意为你解释图片中的数学原理。这涉及到计算机图形学中非常核心的概念——**齐次坐标（Homogeneous Coordinates）**。

### 1. 图片中等式的计算过程

图片展示的是一个标准的矩阵乘法：$A \cdot v$。
根据线性代数中“矩阵乘以向量”的规则（**行乘以列**），我们可以逐行计算：

$$
\begin{bmatrix}
1 & 0 & 0 & x \\
0 & 1 & 0 & y \\
0 & 0 & 1 & z \\
0 & 0 & 0 & 1
\end{bmatrix}
\cdot
\begin{bmatrix}
a \\
b \\
c \\
1
\end{bmatrix}
$$

*   **第 1 行计算（X轴结果）：**
    $(1 \cdot a) + (0 \cdot b) + (0 \cdot c) + (x \cdot 1) = a + 0 + 0 + x = \mathbf{a + x}$

*   **第 2 行计算（Y轴结果）：**
    $(0 \cdot a) + (1 \cdot b) + (0 \cdot c) + (y \cdot 1) = 0 + b + 0 + y = \mathbf{b + y}$

*   **第 3 行计算（Z轴结果）：**
    $(0 \cdot a) + (0 \cdot b) + (1 \cdot c) + (z \cdot 1) = 0 + 0 + c + z = \mathbf{c + z}$

*   **第 4 行计算（W维保留）：**
    $(0 \cdot a) + (0 \cdot b) + (0 \cdot c) + (1 \cdot 1) = 0 + 0 + 0 + 1 = \mathbf{1}$

**结果就是：**
$$
\begin{bmatrix}
a + x \\
b + y \\
c + z \\
1
\end{bmatrix}
$$
这正是把点 $(a, b, c)$ 在空间中移动了 $(x, y, z)$ 的距离。

---

### 2. 为什么要使用 4维矩阵（4x4）来做 3维平移？

这是初学者最常见的疑问：**“我们在 3D 空间里，为什么不用 3x3 矩阵？”**

#### 原因一：线性变换 vs 仿射变换
在线性代数中，**3x3 矩阵只能表示线性变换**（Linear Transformation）。线性变换有一个死规则：**原点 $(0,0,0)$ 变换后必须还在原地**。

如果你用 3x3 矩阵乘以原点：
$$
\begin{bmatrix}
? & ? & ? \\
? & ? & ? \\
? & ? & ?
\end{bmatrix}
\cdot
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$
无论矩阵里填什么数字，结果永远是 0。这意味着 **3x3 矩阵只能做缩放、旋转、错切，唯独做不了“平移”**，因为平移会移动原点。

#### 原因二：统一计算形式（齐次坐标的威力）
为了把“平移”也塞进矩阵乘法的形式里，数学家引入了第 4 个维度（通常称为 $w$），这就是**齐次坐标**。

*   我们将 3维点 $(x, y, z)$ 写成 4维向量 $(x, y, z, 1)$。
*   最后的那个 $1$ 非常关键。在矩阵乘法中，它会与矩阵最后一列的 $x, y, z$（平移量）相乘，从而把平移量“加”到原始坐标上（如第一部分计算所示）。

#### 总结
使用 4x4 矩阵的好处是**归一化**：
它允许我们将 **平移（Translation）**、**旋转（Rotation）** 和 **缩放（Scale）** 全部统一成**矩阵乘法**。

如果没有这个 4D 矩阵，你的代码可能就要写成：
```cpp
// 伪代码：这就很麻烦，不能连乘
v_new = RotationMatrix * v_old + TranslationVector; 
```
用了 4D 矩阵后，所有变换都可以合并成一个矩阵 $M$：
```cpp
// 简洁，高效，GPU 最爱
v_new = M * v_old; 
```

这也是你在 main.cpp 中看到 `glm::mat4`（4x4矩阵）和 `shader.SetUniformMat4f` 的原因。

这段代码实现的是**绕任意轴旋转的旋转矩阵生成算法**。

这是图形学中非常经典的**罗德里格旋转公式 (Rodrigues' Rotation Formula)** 的矩阵形式实现。

我来分步解释这段代码在做什么：

### 1. 归一化旋转轴 (Normalization)
```cpp
float len = length(x, y, z);
if (1.0f != len) { // 正则化
    float recipLen = 1.0f / len;
    x *= recipLen;
    y *= recipLen;
    z *= recipLen;
}
```
*   **目的**：旋转轴向量 $(x, y, z)$ 必须是单位向量（长度为 1）。如果传入的向量长度不是 1，旋转计算会出错（会导致物体变形）。
*   **操作**：计算向量长度，然后算出倒数 `recipLen`，把 x, y, z 分别乘以倒数，这就完成了向量的**标准化（Normalize）**。

### 2. 准备中间变量
```cpp
float nc = 1.0f - c;
float xy = x * y;
// ... (其他乘积组合)
float zs = z * s;
```
这里是为了优化性能，预先计算公式中会反复用到的项。
*   `c` 代表 $\cos(\theta)$ （余弦值，应该在代码前面计算过了）。
*   `s` 代表 $\sin(\theta)$ （正弦值）。
*   `nc` 代表 $1 - \cos(\theta)$，这是罗德里格公式里的关键因子。
*   `x, y, z` 是刚才归一化的旋转轴分量。

### 3.构建旋转矩阵 (填充核心数据)
最下面的一大段赋值是在填充 3x3 的旋转矩阵部分（通常嵌入在 4x4 矩阵的左上角）。

这正是**罗德里格旋转矩阵**的标准形式：

$$
R = \begin{bmatrix}
x^2(1-c)+c & xy(1-c)-zs & xz(1-c)+ys \\
xy(1-c)+zs & y^2(1-c)+c & yz(1-c)-xs \\
xz(1-c)-ys & yz(1-c)+xs & z^2(1-c)+c
\end{bmatrix}
$$

代码对应关系如下（注意 OpenGL 通常是列主序，`rm` 数组下标可能是列优先）：

*   **对角线元素 (Scale-like terms)**：
    *   `rm[offset + 0]` (第1列第1行) = $x \cdot x \cdot (1-c) + c$
    *   `rm[offset + 5]` (第2列第2行) = $y \cdot y \cdot (1-c) + c$
    *   `rm[offset + 10]` (第3列第3行) = $z \cdot z \cdot (1-c) + c$

*   **非对角线元素 (Shear/Rotate terms)**：
    *   `rm[offset + 4]` (第2列第1行) = $x \cdot y \cdot (1-c) - z \cdot s$
    *   ...以此类推

### 总结
这段代码的功能是：
给定一个旋转轴 $(x, y, z)$ 和一个旋转角度（隐含在 $c$ 和 $s$ 中），通过**数学公式直接计算出对应的旋转矩阵**。

如果不使用这段代码，你就只能分别绕 X、Y、Z 轴旋转三次来凑出任意方向，那样会有“万向节死锁”问题，而这个公式可以一次性完美解决**绕任意指定轴旋转**的任务。


这段内容推导的是计算机图形学中核心的**“视图变换矩阵”（View Matrix）**，也就是著名的 `lookAt` 矩阵。

它的核心目的是：**把世界坐标系中的物体，转换到相机（观察者）的坐标系下**。

我用大白话分步解释它的推导逻辑：

### 1. 定义相机的坐标轴（构建基底）

要在数学上描述一个相机，我们需要知道这三件事：
*   **eye**: 相机在哪？（位置）
*   **f (forward)**: 相机看哪？（观察方向）
*   **u (up)**: 相机的头顶朝哪？（上方向）

图片中定义了一个新的坐标系（相机坐标系），它的轴是这样算出来的：
1.  **Z轴**：通常定义为 **$-f$**。为什么是负的？因为在 OpenGL 等标准中，相机看向的是 Z 轴负方向（屏幕里），所以 Z 轴正方向其实是相机背后的方向。
2.  **X轴 (s)**：通过叉乘计算 **$s = u \times f$**。这利用了右手法则，得到一个垂直于“视线”和“头顶”的向量，也就是相机的“右边”。
3.  **Y轴 (u')**：为了确保严格垂直（正交），通常会重新计算 **$u' = f \times s$** （图片里假设 $u$ 已经正交了）。

这样就凑齐了相机坐标系的三个基向量：**$s$ (右)**, **$u$ (上)**, **$-f$ (后)**。

### 2. 变换的本质：逆变换

**观测变换的本质是**：如果你想把相机移到世界原点并摆正，这等价于**把整个世界反向移动并反向旋转**。

*   **原来的世界**：基向量是 $e_1, e_2, e_3$（也就是常规的 X, Y, Z 轴）。
*   **相机的世界**：基向量是 $s, u, -f$。

图片中的公式推导是基于**基变换（Change of Basis）**原理：

#### 第一张公式（矩阵 A）
$$
[s \quad u \quad -f \quad e] = [e_1 \quad e_2 \quad e_3 \quad e_4] \cdot M
$$
这个矩阵 $M$ (也就是图片里的 $A$) 描述的是：**如何用世界坐标轴表示相机坐标轴**。
如果你把相机坐标轴写成列向量拼起来，就是这个矩阵：
$$
\begin{bmatrix}
s_x & u_x & -f_x & 0 \\
s_y & u_y & -f_y & 0 \\
s_z & u_z & -f_z & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$
(注：第四列应该是平移量，图片这里简化讨论旋转部分，设为 0)。

#### 第二张公式（矩阵 A' / 视图矩阵）
但是！我们做渲染时，需要的是**反过来**的过程：我们需要把世界坐标里的点，投影到相机坐标系里。
所以我们需要的是上面那个矩阵的**逆矩阵** ($A^{-1}$)。

因为 $s, u, -f$ 是标准正交基（长度为1且互相垂直），这里有一个线性代数的魔法性质：
**正交矩阵的逆矩阵 = 它的转置矩阵** ($A^{-1} = A^T$)。

所以，图片最后一个公式展示的结果，就是把上面的矩阵行变成列，列变成行：
$$
\begin{bmatrix}
s_x & s_y & s_z & 0 \\
u_x & u_y & u_z & 0 \\
-f_x & -f_y & -f_z & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$
这就是**视图矩阵的旋转部分**。

### 总结

这张图解释了为什么当你调用 `glm::lookAt` 时，生成的矩阵中：
1.  **第一行**是相机的“右向量 (Right)”。
2.  **第二行**是相机的“上向量 (Up)”。
3.  **第三行**是相机的“后向量 (Back)”。

这个矩阵的作用就是：**把世界“掰弯”，让它对齐到相机的视角里去。**